# G1 ablation: bursting and light DoC analysis

Focused longitudinal analysis of the **G1 ablation session**. The primary endpoint is the burst phenotype from the somatic voltage/ephys pipeline; image, change, omission, and pre-omission metrics are included as a lighter functional follow-up.

**Primary comparison:** each registered neuron's G1 value versus its immediately preceding session (B2 in the current 852835 dataset). The median across all pre-G1 sessions is retained as a stability reference.

**Depth groups:** <100 µm, 100–150 µm, >150 µm.

The attached ROI-registration pipeline does not carry an explicit ablation-target flag. By default this notebook therefore analyzes longitudinally registered neurons present in G1. If only a subset of G1 neurons was physically targeted, fill `ABLATION_CELL_IDS` below.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, re, warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import wilcoxon, spearmanr
from statsmodels.stats.multitest import multipletests
from matplotlib.lines import Line2D
from IPython.display import display, HTML

from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.spikes import DETECTOR_VERSION, extract_session_spikes
from vip_slap2_analysis.voltage.analysis import build_analysis_tables, save_analysis_tables
from vip_slap2_analysis.voltage.dataset import build_voltage_session_table, build_voltage_roi_table
from vip_slap2_analysis.behavior.change_detection import build_change_detection_events
from vip_slap2_analysis.voltage.responses import build_single_trial_index

assert DETECTOR_VERSION == "template_v1"

sns.set_style("white")
plt.rcParams.update({
    "legend.fontsize":"large","axes.labelsize":"large","axes.titlesize":"large",
    "xtick.labelsize":"medium","ytick.labelsize":"medium"
})
display(HTML("<style>.container { width:100% !important; }</style>"))


## 1. Configuration


In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
SAVE_PATH = Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots\G1_ablation")
SAVE_PATH.mkdir(parents=True, exist_ok=True)

TARGET_MICE = [852835, 863774]
G1_LABEL = "G1"
PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
TRACE_VARIANT = "dff_robust_f0_trial"
REGISTRATION_FILENAME = "roi_identity_registration.csv"
EXCLUDE_INVALID_ROIS = False  # preserve the ephys-characterization notebook default

# None = every longitudinally registered neuron recorded in G1.
# Otherwise use, e.g. {"852835": ["852835_C000", "852835_C003"]}.
ABLATION_CELL_IDS = None

SPIKE_KWARGS = dict(height_sigma=3.5, template_sigma=4.5, prominence_sigma=0.5)
FEATURE_KWARGS = dict(
    isolation_ms=50.0, waveform_pre_ms=5.0, waveform_post_ms=15.0,
    waveform_peak_refine_ms=2.0, waveform_baseline_window_ms=(-5.0,-1.0),
    waveform_auc_window_ms=(-2.0,10.0), compound_isi_ms=50.0,
    burst_min_spikes=3, spike_sttc_dt_ms=40.0, burst_sttc_dt_ms=40.0,
    spike_count_bin_ms=100.0,
)

GROUP_ORDER = ["<100 µm", "100–150 µm", ">150 µm"]
DEPTH_COLORS = {"<100 µm":"#EBA287", "100–150 µm":"#d1e2b0", ">150 µm":"#7bbcd5"}

BURST_METRICS = {
    "burst_event_rate_hz":"Burst event rate (Hz)",
    "burst_spike_fraction":"Fraction of spikes in bursts",
    "burst_event_fraction":"Fraction of events that are bursts",
    "compound_event_fraction":"Compound event fraction",
    "spike_rate_hz":"Spike rate (Hz)",
    "median_burst_duration_ms":"Median burst duration (ms)",
    "median_burst_n_spikes":"Median spikes / burst",
}
PRIMARY_BURST_METRICS = [
    "burst_event_rate_hz","burst_spike_fraction",
    "compound_event_fraction","spike_rate_hz",
]

IMAGE_WINDOW_S = (0.0, 0.25)
IMAGE_BASELINE_S = (-0.25, 0.0)
CHANGE_RESPONSE_WINDOW_S = (0.0, 0.25)
OMISSION_MATCH_WINDOW_S = (0.0, 0.50)
PRE_OMISSION_RAMP_EARLY_S = (-0.75, -0.25)
PRE_OMISSION_RAMP_LATE_S = (-0.25, 0.0)
MIN_TRIALS_PER_IMAGE = 5
N_MATCHED_CONTROLS = 5
MATCH_POSITION_TOLERANCE = 2
EXPECTED_F0_SMOOTH_SEC = 60.0

DOC_METRICS = {
    "mean_image_delta_dff":"Image response ΔF/F",
    "change_minus_matched_same_image_dff":"Change − matched image ΔF/F",
    "omission_minus_matched_dff":"Omission − matched cycle ΔF/F",
    "pre_omission_ramp_dff":"Pre-omission ramp ΔF/F",
}
COUPLING_BURST_METRIC = "burst_spike_fraction"


## 2. Ephys processing and longitudinal G1 cohort


In [ ]:
def session_label(row):
    if pd.notna(row.get("image_set")) and pd.notna(row.get("image_set_day_index")):
        return f"{row['image_set']}{int(row['image_set_day_index'])}"
    return str(row.get("session_type", row["session_id"]))

def session_depth(row, dmd):
    for key in (f"dmd{dmd}_depth", f"dmd{dmd}_depth_um"):
        if key in row and pd.notna(row[key]): return float(row[key])
    metadata = row.get("metadata", {}) if isinstance(row.get("metadata", {}), dict) else {}
    return float(metadata.get(f"dmd{dmd}_depth", np.nan))

def depth_group(depth):
    if not np.isfinite(depth): return np.nan
    return "<100 µm" if depth < 100 else ("100–150 µm" if depth <= 150 else ">150 µm")

def target_mask(df):
    if ABLATION_CELL_IDS is None: return np.ones(len(df), bool)
    if isinstance(ABLATION_CELL_IDS, dict):
        return np.array([
            str(gid) in set(map(str, ABLATION_CELL_IDS.get(str(sid), ABLATION_CELL_IDS.get(int(sid) if str(sid).isdigit() else sid, []))))
            for sid, gid in zip(df["subject_id"], df["global_cell_id"])
        ], bool)
    allowed = set(map(str, ABLATION_CELL_IDS))
    return df["global_cell_id"].astype(str).isin(allowed).to_numpy()

registry = VIPSessionRegistry.from_basepath(BASE_PATH)
raw = registry.sessions(subject_ids=TARGET_MICE, paradigms=PARADIGMS, exclude_session_types=EXCLUDE_SESSION_TYPES).copy()
raw["session_id"] = raw["session_id"].astype(str)
raw["subject_id"] = raw["subject_id"].astype(str)
raw["session_datetime"] = pd.to_datetime(
    raw["session_id"].str.extract(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})")[0],
    format="%Y-%m-%d_%H-%M-%S", errors="coerce"
)
raw = raw.sort_values(["subject_id","session_datetime"]).reset_index(drop=True)
raw["session_order"] = raw.groupby("subject_id").cumcount()
raw["session_label"] = raw.apply(session_label, axis=1)

g1_subjects = sorted(raw.loc[raw["session_label"].astype(str).eq(G1_LABEL), "subject_id"].unique())
if not g1_subjects: raise RuntimeError(f"No {G1_LABEL} sessions found.")

g1_order = (raw[raw["session_label"].astype(str).eq(G1_LABEL)]
            .groupby("subject_id")["session_order"].max().to_dict())
raw = raw[raw["subject_id"].isin(g1_subjects)].copy()
raw = raw[raw.apply(lambda r: r["session_order"] <= g1_order[r["subject_id"]], axis=1)].copy()

rows = []
for _, row in raw.iterrows():
    asset = registry.resolve_assets(row)
    trace_h5 = Path(asset.derived_dir) / "voltage" / f"voltage_session_traces_{TRACE_VARIANT}.h5"
    if not trace_h5.exists():
        warnings.warn(f"{asset.session_id}: missing {trace_h5.name}; skipped"); continue
    rows.append({
        "subject_id":str(asset.subject_id),"session_id":str(asset.session_id),
        "session_label":str(row["session_label"]),"session_order":int(row["session_order"]),
        "session_type":row.get("session_type",""),
        "dmd1_depth_um":session_depth(row,1),"dmd2_depth_um":session_depth(row,2),
        "trace_h5":trace_h5,"derived_dir":Path(asset.derived_dir),
        "subject_dir":Path(asset.session_dir).parent,
    })

sessions = pd.DataFrame(rows).sort_values(["subject_id","session_order"]).reset_index(drop=True)
display(sessions[["subject_id","session_id","session_label","session_order","dmd1_depth_um","dmd2_depth_um"]])


In [ ]:
roi_rows = []
for session in sessions.itertuples(index=False):
    with h5py.File(session.trace_h5, "r") as h5:
        for dmd_key in sorted(k for k in h5 if k.startswith("DMD")):
            dmd = int(dmd_key.replace("DMD","")); group = h5[dmd_key]
            n_time = len(group["timebase_sec"]); dff = group["dff"]
            n_rois = dff.shape[0] if dff.shape[-1] == n_time else dff.shape[1]
            valid = group["valid_rois_mask"][:] if "valid_rois_mask" in group else np.ones(n_rois, bool)
            if len(valid) != n_rois: valid = np.ones(n_rois, bool)
            depth = getattr(session, f"dmd{dmd}_depth_um")
            for roi in range(n_rois):
                roi_rows.append(dict(
                    subject_id=session.subject_id,session_id=session.session_id,
                    session_label=session.session_label,session_order=session.session_order,
                    session_type=session.session_type,dmd=dmd,roi=roi,depth_um=depth,valid_roi=bool(valid[roi])
                ))
rois = pd.DataFrame(roi_rows)

regs = []
for subject_id, q in sessions.groupby("subject_id"):
    path = Path(q.iloc[0]["subject_dir"]) / REGISTRATION_FILENAME
    if not path.exists(): warnings.warn(f"No manual ROI registry for {subject_id}: {path}"); continue
    t = pd.read_csv(path, dtype={"subject_id":str,"session_id":str,"global_cell_id":str})
    t["subject_id"] = str(subject_id)
    keep = [c for c in ["subject_id","session_id","dmd","roi","global_cell_id","excluded","confidence","notes"] if c in t]
    regs.append(t[keep].drop_duplicates(["subject_id","session_id","dmd","roi"], keep="last"))
if regs:
    rois = rois.merge(pd.concat(regs, ignore_index=True), on=["subject_id","session_id","dmd","roi"], how="left")

if "global_cell_id" not in rois: rois["global_cell_id"] = ""
if "excluded" not in rois: rois["excluded"] = False
rois["global_cell_id"] = rois["global_cell_id"].fillna("").replace("nan","")
rois["excluded"] = rois["excluded"].fillna(False).map(
    lambda x: x if isinstance(x,(bool,np.bool_)) else str(x).strip().lower() in {"true","1","yes","y"}
)
rois["manually_registered"] = rois["global_cell_id"].ne("")
rois["cell_id"] = np.where(
    rois["manually_registered"], rois["global_cell_id"],
    rois["session_id"]+"_DMD"+rois["dmd"].astype(str)+"_ROI"+rois["roi"].astype(str)
)
rois["included"] = ~rois["excluded"] & (~EXCLUDE_INVALID_ROIS | rois["valid_roi"])
rois["depth_group"] = rois["depth_um"].map(depth_group)
display(rois.loc[rois["included"], ["subject_id","session_label","dmd","roi","global_cell_id","depth_um","depth_group"]].head())


In [ ]:
tables = []
for session in sessions.itertuples(index=False):
    print(f"Spikes: {session.subject_id} {session.session_label}")
    sr = rois[(rois["session_id"] == session.session_id) & rois["included"]]
    lookup = {int(dmd): g["roi"].astype(int).tolist() for dmd,g in sr.groupby("dmd")}
    t = extract_session_spikes(session.trace_h5, rois=lookup, **SPIKE_KWARGS)
    t.insert(0, "session_id", session.session_id); tables.append(t)

spikes = pd.concat(tables, ignore_index=True)
spikes = spikes.merge(
    rois[["subject_id","session_id","session_label","session_order","session_type","dmd","roi",
          "depth_um","cell_id","global_cell_id","manually_registered"]],
    on=["session_id","dmd","roi"], how="left"
)
print(f"{len(spikes):,} spikes")

analysis_tables = build_analysis_tables(sessions, rois, spikes, **FEATURE_KWARGS)
roi_features = analysis_tables["roi_features"]
save_analysis_tables(sessions, analysis_tables, subdir=Path("voltage")/"g1_ablation", parameters=FEATURE_KWARGS)
print(f"{len(roi_features):,} ROI × session feature rows")


In [ ]:
def longitudinal_registered(df):
    q = df.copy()
    q["global_cell_id"] = q["global_cell_id"].astype(str)
    q = q[q["manually_registered"].astype(bool) & ~q["global_cell_id"].isin(["","nan","None"])].copy()
    g1_ids = set(q.loc[q["session_label"].astype(str).eq(G1_LABEL), "global_cell_id"])
    q = q[q["global_cell_id"].isin(g1_ids)].copy()
    q = q[target_mask(q)].copy()
    q["neuron_id"] = q["subject_id"].astype(str)+":"+q["global_cell_id"].astype(str)
    neuron_depth = q.groupby("neuron_id")["depth_um"].median()
    q["neuron_depth_um"] = q["neuron_id"].map(neuron_depth)
    q["depth_group"] = q["neuron_depth_um"].map(depth_group)
    return q

def make_g1_pairs(df, metrics):
    q = longitudinal_registered(df)
    rows = []
    for subject_id, s in q.groupby("subject_id"):
        g1 = s[s["session_label"].astype(str).eq(G1_LABEL)]
        if g1.empty: continue
        g1_ord = int(g1["session_order"].max())
        prev_ord = int(s.loc[s["session_order"] < g1_ord, "session_order"].max())
        prev_label = str(s.loc[s["session_order"].eq(prev_ord), "session_label"].iloc[0])
        pre = s[s["session_order"] < g1_ord]
        for neuron_id, c in s.groupby("neuron_id"):
            cg1 = c[c["session_order"].eq(g1_ord)]
            cprev = c[c["session_order"].eq(prev_ord)]
            cpre = pre[pre["neuron_id"].eq(neuron_id)]
            if cg1.empty or cprev.empty: continue
            row = dict(
                subject_id=str(subject_id),neuron_id=neuron_id,
                global_cell_id=str(cg1["global_cell_id"].iloc[0]),
                previous_label=prev_label,depth_um=float(c["neuron_depth_um"].median()),
                depth_group=str(c["depth_group"].iloc[0]),n_pre_sessions=int(cpre["session_id"].nunique())
            )
            for m in metrics:
                vg = float(pd.to_numeric(cg1[m],errors="coerce").median())
                vp = float(pd.to_numeric(cprev[m],errors="coerce").median())
                vm = float(pd.to_numeric(cpre[m],errors="coerce").median())
                row.update({f"{m}_g1":vg,f"{m}_previous":vp,f"{m}_pre_median":vm,
                            f"{m}_delta_previous":vg-vp,f"{m}_delta_pre_median":vg-vm})
            rows.append(row)
    return pd.DataFrame(rows)

burst_long = longitudinal_registered(roi_features)
burst_pairs = make_g1_pairs(roi_features, list(BURST_METRICS))
coverage = burst_pairs.groupby(["subject_id","depth_group"], observed=True).agg(
    n_cells=("neuron_id","nunique"),previous_session=("previous_label","first"),
    median_pre_sessions=("n_pre_sessions","median")
).reset_index()
display(coverage)


## 3. Burst phenotype

The key question is whether G1 reduces **bursting itself**, rather than simply reducing total spike rate. Accordingly, burst event rate is shown alongside burst-spike fraction / burst-event fraction and overall spike rate.


In [ ]:
def finish_axis(ax):
    sns.despine(ax=ax)
    ax.tick_params(axis="both", labelsize=10)
    for sp in ax.spines.values():
        sp.set_linewidth(2)

session_order = (
    sessions[["session_label", "session_order"]]
    .drop_duplicates()
    .groupby("session_label", as_index=False)["session_order"].median()
    .sort_values("session_order")
)

SESSION_ORDER = [
    x for x in session_order["session_label"]
    if x in set(burst_long["session_label"].astype(str))
]
xpos = {s: i for i, s in enumerate(SESSION_ORDER)}

titles = ['Burst rate','Fraction of spikes in bursts','Compound event fraction','Spike rate']

for i,m in enumerate(PRIMARY_BURST_METRICS):
    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
    ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

    # Individual neurons
    for neuron_id, c in burst_long.groupby("neuron_id"):
        c = c.sort_values("session_order")
        g = c["depth_group"].iloc[0]

        x = c["session_label"].map(xpos).astype(float).to_numpy()

        ax.plot(
            x, c[m],
            color=DEPTH_COLORS[g],
            alpha=.25,
            lw=1,
            marker='.'
        )

    # Depth-group medians
    for g in GROUP_ORDER:
        med = (
            burst_long[burst_long["depth_group"].eq(g)]
            .groupby("session_label", observed=True)[m]
            .median()
            .reindex(SESSION_ORDER)
        )

        good = med.notna().to_numpy()
        xx = np.arange(len(SESSION_ORDER))[good]

        ax.plot(
            xx,
            med.to_numpy()[good],
            color=DEPTH_COLORS[g],
            lw=3,
            marker="o",
            ms=5,
            mec="black",
            mew=.6,
            label=g,
        )

    # Mark ablation session
    if G1_LABEL in xpos:
        ax.axvline(
            5.5,
            ls="--",
            lw=1.5,
            alpha=0.5,
            color="r",label='ablation',zorder=0
        )
    ax.axvline(
            2.5,
            ls="--",
            lw=1.5,
            color="k",zorder=0
        )
    ax.set_xticks(range(len(SESSION_ORDER)))
    ax.set_xticklabels(SESSION_ORDER)
    ax.set_xlabel("Session")
    ax.set_ylabel(BURST_METRICS[m])
    ax.set_title(titles[i])
    ax.legend(
        frameon=False,
        title="Depth",
        fontsize=8,loc='best'
    )

    finish_axis(ax)
    fig.tight_layout()

    save_figure(
        fig,
        os.path.join(SAVE_PATH, f"G1_burst_longitudinal_{m}"),
        formats=[".pdf", ".png"],
        dpi=300,
    )

    plt.show()

In [ ]:
xpos

In [ ]:
def paired_wilcoxon_table(pair_df, metrics, reference="previous"):
    rows=[]
    for m in metrics:
        for g in ["all"]+GROUP_ORDER:
            q=pair_df if g=="all" else pair_df[pair_df["depth_group"].eq(g)]
            a=pd.to_numeric(q[f"{m}_{reference}"],errors="coerce")
            b=pd.to_numeric(q[f"{m}_g1"],errors="coerce")
            good=a.notna()&b.notna(); a=a[good].to_numpy(); b=b[good].to_numpy()
            if len(a)<3: p=np.nan
            else:
                try: p=wilcoxon(a,b,alternative="two-sided").pvalue
                except ValueError: p=1.0
            rows.append(dict(metric=m,depth_group=g,n_pairs=len(a),
                             median_reference=np.nanmedian(a) if len(a) else np.nan,
                             median_g1=np.nanmedian(b) if len(b) else np.nan,
                             median_delta=np.nanmedian(b-a) if len(a) else np.nan,p_raw=p))
    out=pd.DataFrame(rows); out["p_holm"]=np.nan
    for m in metrics:
        ix=out.index[(out["metric"].eq(m))&(out["depth_group"].isin(GROUP_ORDER))&out["p_raw"].notna()]
        if len(ix): out.loc[ix,"p_holm"]=multipletests(out.loc[ix,"p_raw"],method="holm")[1]
    return out

burst_stats = paired_wilcoxon_table(burst_pairs, list(BURST_METRICS), reference="previous")
display(burst_stats[burst_stats["metric"].isin(PRIMARY_BURST_METRICS)])


In [ ]:
def plot_delta_grid(pair_df, metrics, labels, suffix="_delta_previous", filename=None, figsize=(8,6)):
    fig,axs=plt.subplots(2,2,figsize=figsize); axs=np.asarray(axs).ravel()
    for ax,m in zip(axs,metrics):
        col=m+suffix; q=pair_df.dropna(subset=[col,"depth_group"]).copy()
        ax.axhline(0,color="0.4",lw=1,ls="--")
        sns.violinplot(data=q,x="depth_group",y=col,order=GROUP_ORDER,palette=DEPTH_COLORS,
                       inner=None,cut=0,width=.65,linewidth=1.3,ax=ax)
        for coll in ax.collections[:len(GROUP_ORDER)]: coll.set_alpha(.22)
        sns.stripplot(data=q,x="depth_group",y=col,order=GROUP_ORDER,palette=DEPTH_COLORS,
                      jitter=.14,size=5,edgecolor="black",linewidth=.35,ax=ax)
        meds=q.groupby("depth_group",observed=True)[col].median().reindex(GROUP_ORDER)
        ax.scatter(np.arange(len(GROUP_ORDER)),meds,s=55,facecolor="white",edgecolor="black",zorder=5)
        ax.set_xlabel(""); ax.set_ylabel("G1 − previous"); ax.set_title(labels[m],fontsize=11); finish_axis(ax)
    for ax in axs[len(metrics):]: ax.axis("off")
    fig.tight_layout()
    if filename: save_figure(fig, os.path.join(SAVE_PATH,filename), formats=[".pdf"], dpi=300)
    return fig

plot_delta_grid(burst_pairs, PRIMARY_BURST_METRICS, BURST_METRICS, filename="G1_burst_delta_by_depth")


In [ ]:
# Primary endpoint as explicit paired previous-session → G1 trajectories.
m="burst_event_rate_hz"
fig,axs=plt.subplots(1,3,figsize=(8.2,3.2),sharey=True)
for ax,g in zip(axs,GROUP_ORDER):
    q=burst_pairs[burst_pairs["depth_group"].eq(g)].dropna(subset=[f"{m}_previous",f"{m}_g1"])
    for r in q.itertuples(index=False):
        ax.plot([0,1],[getattr(r,f"{m}_previous"),getattr(r,f"{m}_g1")],color=DEPTH_COLORS[g],alpha=.45,lw=1)
    ax.scatter(np.zeros(len(q)),q[f"{m}_previous"],s=22,color=DEPTH_COLORS[g],edgecolor="black",linewidth=.3)
    ax.scatter(np.ones(len(q)),q[f"{m}_g1"],s=22,color=DEPTH_COLORS[g],edgecolor="black",linewidth=.3)
    ax.set_xticks([0,1]); ax.set_xticklabels([q["previous_label"].iloc[0] if len(q) else "previous",G1_LABEL])
    ax.set_title(f"{g}\nn={len(q)}",fontsize=10); finish_axis(ax)
axs[0].set_ylabel(BURST_METRICS[m])
fig.tight_layout()
save_figure(fig, os.path.join(SAVE_PATH,"G1_burst_rate_paired"), formats=[".pdf"], dpi=300)


In [ ]:
# Is the burst phenotype larger than expected from the change in total firing?
fig,axs=plt.subplots(1,2,figsize=(7.2,3.2))
for ax,m in zip(axs,["burst_event_rate_hz","burst_spike_fraction"]):
    x="spike_rate_hz_delta_previous"; y=f"{m}_delta_previous"
    q=burst_pairs.dropna(subset=[x,y,"depth_group"])
    for g in GROUP_ORDER:
        gg=q[q["depth_group"].eq(g)]
        ax.scatter(gg[x],gg[y],s=35,color=DEPTH_COLORS[g],edgecolor="black",linewidth=.4,label=g)
    ax.axhline(0,color=".5",lw=1,ls="--"); ax.axvline(0,color=".5",lw=1,ls="--")
    rho,p=spearmanr(q[x],q[y]) if len(q)>=4 else (np.nan,np.nan)
    ax.set_xlabel("G1 − previous spike rate (Hz)"); ax.set_ylabel(f"G1 − previous\n{BURST_METRICS[m]}")
    ax.text(.03,.97,f"Spearman ρ={rho:.2f}, p={p:.3g}\nn={len(q)}",transform=ax.transAxes,va="top",fontsize=8)
    finish_axis(ax)
axs[0].legend(frameon=False,title="Depth",fontsize=8)
fig.tight_layout()
save_figure(fig, os.path.join(SAVE_PATH,"G1_burst_specificity"), formats=[".pdf"], dpi=300)


## 4. Light DoC-specific analysis

These metrics intentionally reuse the event definitions from the DoC notebook rather than adding a second response model:

- image: 0–250 ms minus −250–0 ms;
- change: change response minus nearby matched ordinary presentations of the same image;
- omission: 0–500 ms omission cycle minus matched expected-image cycles;
- pre-omission ramp: mean(−250–0 ms) minus mean(−750–−250 ms).

The same G1-versus-immediately-previous registered-cell comparison is used.


In [ ]:
doc_sessions = build_voltage_session_table(
    registry, subject_ids=[int(x) for x in g1_subjects], paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES, trace_variant=TRACE_VARIANT,
    expected_f0_smooth_sec=EXPECTED_F0_SMOOTH_SEC
).copy()
doc_sessions["subject_id"]=doc_sessions["subject_id"].astype(str)
doc_sessions["session_id"]=doc_sessions["session_id"].astype(str)

# Use exactly the sessions retained by the ephys side.
doc_sessions = doc_sessions[doc_sessions["session_id"].isin(set(sessions["session_id"]))].copy()
doc_rois = build_voltage_roi_table(
    doc_sessions, registration_filename=REGISTRATION_FILENAME,
    exclude_invalid_rois=EXCLUDE_INVALID_ROIS
)
doc_rois["subject_id"]=doc_rois["subject_id"].astype(str)
doc_rois["session_id"]=doc_rois["session_id"].astype(str)
doc_rois["depth_um"]=pd.to_numeric(doc_rois["depth_um"],errors="coerce")
doc_rois["depth_group"]=doc_rois["depth_um"].map(depth_group)

events=build_change_detection_events(doc_sessions)
trial_index=build_single_trial_index(doc_sessions,events)
print(f"{len(doc_sessions)} DoC sessions · {doc_rois['included'].sum()} included ROI observations")


In [ ]:
def decode_strings(values):
    return np.asarray([x.decode() if isinstance(x,(bytes,np.bytes_)) else str(x) for x in np.asarray(values).reshape(-1)])

def h5_roi_axis(group,dmd,source_roi):
    ids=decode_strings(group["roi_ids"][:]); label=f"DMD{int(dmd)}_ROI{int(source_roi)}"
    hits=np.flatnonzero(ids==label)
    if not len(hits):
        parsed=np.array([int(re.findall(r"\d+",x)[-1]) for x in ids],int); hits=np.flatnonzero(parsed==int(source_roi))
    if not len(hits): raise KeyError(f"{label} absent from H5 kept ROI axis")
    return int(hits[0])

def reconcile_timebase(t,n):
    t=np.asarray(t,float).reshape(-1)
    if len(t)==n:return t
    if len(t)<2:raise ValueError(f"Cannot reconcile {len(t)} time samples to {n} trace samples")
    dt=float(np.nanmedian(np.diff(t))); zero=min(int(np.nanargmin(np.abs(t))),n-1)
    return (np.arange(n,dtype=float)-zero)*dt

def window_slice(t,window):
    t=np.asarray(t,float); a,b=map(float,window)
    i0=int(np.searchsorted(t,a,"left")); i1=int(np.searchsorted(t,b,"left"))
    if i1<=i0:raise ValueError(f"Window {window} absent from timebase")
    return slice(i0,i1)

def window_mean(y,t,window,axis=-1):
    return np.nanmean(np.asarray(y,float)[...,window_slice(t,window)],axis=axis)

def shifted_window_means(traces,t,offsets,window):
    out=np.full(len(traces),np.nan,float)
    for i,off in enumerate(np.asarray(offsets,float)):
        if np.isfinite(off):out[i]=window_mean(traces[i],t,(window[0]+off,window[1]+off))
    return out

def indexed_event_table(session_id,dmd,event_type):
    sid=str(session_id)
    idx=trial_index[
        trial_index["session_id"].astype(str).eq(sid)&
        trial_index["dmd"].astype(int).eq(int(dmd))&
        trial_index["event_type"].astype(str).eq(str(event_type))
    ].copy()
    if "matched" in idx:idx=idx[idx["matched"].astype(bool)]
    idx=idx.rename(columns={"onset_sec":"stored_onset_sec","image_name":"stored_image_name"})
    ev=events[events["session_id"].astype(str).eq(sid)].copy()
    needed=["event_id","onset_sec","image_label","is_change","is_omission","sequence_position_expected"]
    for c in needed:
        if c not in ev:ev[c]=np.nan
    ev=ev[needed].rename(columns={"onset_sec":"cycle_onset_sec","image_label":"event_image_label"})
    return idx.merge(ev,on="event_id",how="left",validate="many_to_one").sort_values("trial_index").reset_index(drop=True)

_IMAGE_CONTROL_CACHE={}
def load_image_control_table(h5,session_id,dmd,source_roi,axis):
    key=(str(session_id),int(dmd),int(source_roi))
    if key in _IMAGE_CONTROL_CACHE:return _IMAGE_CONTROL_CACHE[key]
    idx=indexed_event_table(session_id,dmd,"image")
    idx=idx[(~idx["is_change"].fillna(False).astype(bool))&(~idx["is_omission"].fillna(False).astype(bool))].copy()
    stored_t=np.asarray(h5["timebase_sec/image"][:],float); parts=[]
    for path,q in idx.groupby("dataset_path",sort=False):
        q=q.sort_values("trial_index").copy(); rows=q["trial_index"].astype(int).to_numpy()
        ds=h5[str(path)]; t=reconcile_timebase(stored_t,int(ds.shape[-1])); arr=np.asarray(ds[rows,int(axis),:],float)
        q["image_dff"]=window_mean(arr,t,IMAGE_WINDOW_S)
        q["cycle_dff"]=window_mean(arr,t,OMISSION_MATCH_WINDOW_S)
        parts.append(q)
    out=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
    _IMAGE_CONTROL_CACHE[key]=out
    return out

def nearest_control_mean(label,onset_sec,sequence_position,controls,value_col):
    if controls.empty or pd.isna(label):return np.nan
    q=controls[controls["event_image_label"].astype(str).eq(str(label))].copy()
    pos0=pd.to_numeric(sequence_position,errors="coerce")
    if np.isfinite(pos0) and len(q):
        pos=pd.to_numeric(q["sequence_position_expected"],errors="coerce")
        near=np.abs(pos-float(pos0))<=MATCH_POSITION_TOLERANCE
        if near.any():q=q[near]
    if not len(q):return np.nan
    q["distance"]=np.abs(pd.to_numeric(q["cycle_onset_sec"],errors="coerce")-float(onset_sec))
    return float(np.nanmean(pd.to_numeric(q.nsmallest(N_MATCHED_CONTROLS,"distance")[value_col],errors="coerce")))


In [ ]:
doc_rows=[]
for session in doc_sessions.itertuples(index=False):
    sid=str(session.session_id)
    sr=doc_rois[doc_rois["included"].astype(bool)&doc_rois["session_id"].astype(str).eq(sid)]
    print(f"DoC: {session.subject_id} {session.session_label}")
    with h5py.File(session.single_trial_h5,"r") as h5:
        for r in sr.itertuples(index=False):
            group=h5[f"DMD{int(r.dmd)}"]; axis=h5_roi_axis(group,int(r.dmd),int(r.roi))
            meta=dict(subject_id=str(r.subject_id),session_id=sid,session_label=str(r.session_label),
                      session_order=int(r.session_order),dmd=int(r.dmd),roi=int(r.roi),
                      global_cell_id=str(getattr(r,"global_cell_id","")),
                      manually_registered=bool(getattr(r,"manually_registered",False)),
                      depth_um=float(r.depth_um),depth_group=str(r.depth_group))

            # Image response.
            stored_t=np.asarray(h5["timebase_sec/image"][:],float); image_delta=[]
            for key in group["image_identity"]:
                sub=group["image_identity"][key]
                if int(sub["traces"].shape[0])<MIN_TRIALS_PER_IMAGE:continue
                t=reconcile_timebase(stored_t,int(sub["traces"].shape[-1]))
                arr=np.asarray(sub["traces"][:,axis,:],float)
                image_delta.extend((window_mean(arr,t,IMAGE_WINDOW_S)-window_mean(arr,t,IMAGE_BASELINE_S)).tolist())

            controls=load_image_control_table(h5,sid,int(r.dmd),int(r.roi),axis)

            # Change response.
            sub=group["change"]; traces=np.asarray(sub["traces"][:,axis,:],float)
            t=reconcile_timebase(np.asarray(h5["timebase_sec/change"][:],float),traces.shape[-1])
            idx=indexed_event_table(sid,int(r.dmd),"change")
            if len(idx)!=len(traces):raise ValueError(f"Change trial mismatch: {sid} DMD{r.dmd}")
            offsets=pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy()-pd.to_numeric(idx["stored_onset_sec"],errors="coerce").to_numpy()
            change=shifted_window_means(traces,t,offsets,CHANGE_RESPONSE_WINDOW_S)
            matched_change=np.array([
                nearest_control_mean(x.event_image_label,x.cycle_onset_sec,x.sequence_position_expected,controls,"image_dff")
                for x in idx.itertuples(index=False)
            ])

            # Omission response and pre-omission ramp.
            sub=group["omission"]; traces=np.asarray(sub["traces"][:,axis,:],float)
            t=reconcile_timebase(np.asarray(h5["timebase_sec/omission"][:],float),traces.shape[-1])
            idx=indexed_event_table(sid,int(r.dmd),"omission")
            if len(idx)!=len(traces):raise ValueError(f"Omission trial mismatch: {sid} DMD{r.dmd}")
            offsets=pd.to_numeric(idx["cycle_onset_sec"],errors="coerce").to_numpy()-pd.to_numeric(idx["stored_onset_sec"],errors="coerce").to_numpy()
            omission=shifted_window_means(traces,t,offsets,OMISSION_MATCH_WINDOW_S)
            early=shifted_window_means(traces,t,offsets,PRE_OMISSION_RAMP_EARLY_S)
            late=shifted_window_means(traces,t,offsets,PRE_OMISSION_RAMP_LATE_S)
            matched_omission=np.array([
                nearest_control_mean(x.event_image_label,x.cycle_onset_sec,x.sequence_position_expected,controls,"cycle_dff")
                for x in idx.itertuples(index=False)
            ])

            doc_rows.append({
                **meta,
                "mean_image_delta_dff":float(np.nanmean(image_delta)) if len(image_delta) else np.nan,
                "change_minus_matched_same_image_dff":float(np.nanmean(change-matched_change)),
                "omission_minus_matched_dff":float(np.nanmean(omission-matched_omission)),
                "pre_omission_ramp_dff":float(np.nanmean(late-early)),
                "n_image_trials":len(image_delta),"n_changes":len(change),"n_omissions":len(omission),
            })

doc_metrics=pd.DataFrame(doc_rows)
doc_pairs=make_g1_pairs(doc_metrics,list(DOC_METRICS))
display(doc_pairs.head())


In [ ]:
doc_stats=paired_wilcoxon_table(doc_pairs,list(DOC_METRICS),reference="previous")
display(doc_stats)

plot_delta_grid(
    doc_pairs,list(DOC_METRICS),DOC_METRICS,
    filename="G1_DoC_delta_by_depth",figsize=(8,6)
)


## 5. Does the loss of bursting predict the functional G1 effect?

This last panel links the ephys phenotype to DoC changes within the **same registered neurons**. `burst_spike_fraction` is the default coupling metric because it is less mechanically tied to total firing rate than burst-event rate; change `COUPLING_BURST_METRIC` above if desired.


In [ ]:
xcol=f"{COUPLING_BURST_METRIC}_delta_previous"
e=burst_pairs[["subject_id","global_cell_id","depth_group",xcol]].copy()
d=doc_pairs[["subject_id","global_cell_id"]+[f"{m}_delta_previous" for m in DOC_METRICS]].copy()
coupled=e.merge(d,on=["subject_id","global_cell_id"],how="inner")

fig,axs=plt.subplots(2,2,figsize=(8,6)); axs=axs.ravel()
for ax,(m,label) in zip(axs,DOC_METRICS.items()):
    y=f"{m}_delta_previous"; q=coupled.dropna(subset=[xcol,y])
    for g in GROUP_ORDER:
        gg=q[q["depth_group"].eq(g)]
        ax.scatter(gg[xcol],gg[y],s=38,color=DEPTH_COLORS[g],edgecolor="black",linewidth=.4,label=g)
    ax.axhline(0,color=".5",lw=1,ls="--"); ax.axvline(0,color=".5",lw=1,ls="--")
    rho,p=spearmanr(q[xcol],q[y]) if len(q)>=4 else (np.nan,np.nan)
    ax.set_xlabel(f"Δ {BURST_METRICS[COUPLING_BURST_METRIC]}")
    ax.set_ylabel(f"Δ {label}")
    ax.text(.03,.97,f"ρ={rho:.2f}, p={p:.3g}, n={len(q)}",transform=ax.transAxes,va="top",fontsize=8)
    finish_axis(ax)
axs[0].legend(frameon=False,title="Depth",fontsize=8)
fig.tight_layout()
save_figure(fig, os.path.join(SAVE_PATH,"G1_burst_DoC_coupling"), formats=[".pdf"], dpi=300)


## 6. Export analysis tables

Interpret cell-level p-values as **within-animal exploratory evidence**, not population-level inference, because the current G1 dataset contains only the animal(s) that actually reached G1. The most important checks are therefore effect size, consistency across tracked neurons, depth specificity, stability across the pre-G1 trajectory, and whether burst fractions change even when spike rate does not.


In [ ]:
burst_pairs.to_csv(SAVE_PATH/"G1_ephys_paired_cells.csv",index=False)
burst_stats.to_csv(SAVE_PATH/"G1_ephys_paired_stats.csv",index=False)
doc_pairs.to_csv(SAVE_PATH/"G1_DoC_paired_cells.csv",index=False)
doc_stats.to_csv(SAVE_PATH/"G1_DoC_paired_stats.csv",index=False)
coupled.to_csv(SAVE_PATH/"G1_ephys_DoC_coupled_cells.csv",index=False)

summary = pd.concat([
    burst_stats.assign(domain="ephys"),
    doc_stats.assign(domain="DoC"),
],ignore_index=True)
display(summary)
print(f"Saved tables and figures to: {SAVE_PATH}")
